# 02 - Data Preprocessing & Cleaning
## CSE-CIC-IDS2018

**Input:** `file_100.csv`, `file_75.csv`, `file_50.csv`, `file_25.csv` (dari Notebook 01)

**Langkah Pembersihan:**
1. Hapus/impute NaN & Inf (Inf → max kolom)
2. Buang kolom: Timestamp, Dst Port, Protocol (opsional)
3. Hapus kolom dengan varians nol
4. Scaling fitur numerik (StandardScaler)
5. Encode label string → angka (LabelEncoder)

**Output:** `cleaned_100.pkl`, `cleaned_75.pkl`, `cleaned_50.pkl`, `cleaned_25.pkl`

In [ ]:
import pandas as pd
import numpy as np
import os, gc, warnings, pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
warnings.filterwarnings('ignore')

# Read setting
setting = {}
with open('setting.txt', 'r') as f:
    for line in f:
        key, val = line.strip().split('=')
        setting[key] = val

DATA_DIR = '../data/'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Kolom yang akan dibuang
DROP_COLS = ['Timestamp', 'Dst Port', 'Protocol', 'Flow ID', 'Src IP', 'Dst IP', 'Src Port']

# File yang akan diproses
INPUT_FILES = ['file_100.csv', 'file_75.csv', 'file_50.csv', 'file_25.csv']
OUTPUT_FILES = ['cleaned_100.pkl', 'cleaned_75.pkl', 'cleaned_50.pkl', 'cleaned_25.pkl']

print(f'Data dir: {DATA_DIR}')
print(f'Input files: {INPUT_FILES}')
print(f'Columns to drop: {DROP_COLS}')

## 1. Fungsi Preprocessing

In [ ]:
def preprocess_dataset(df, scaler=None, label_encoder=None, zero_var_cols=None, fit=True):
    """
    Preprocessing pipeline untuk dataset CIC-IDS2018.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    scaler : StandardScaler or None
        Jika fit=False, gunakan scaler yang sudah di-fit
    label_encoder : LabelEncoder or None
        Jika fit=False, gunakan encoder yang sudah di-fit
    zero_var_cols : list or None
        Kolom zero-variance dari dataset referensi (file_100)
    fit : bool
        True = fit_transform, False = transform only
    
    Returns:
    --------
    dict with keys: X, y, scaler, label_encoder, zero_var_cols, label_mapping, feature_names, stats
    """
    stats = {}
    stats['original_shape'] = df.shape
    print(f'  Original shape: {df.shape}')
    
    # --- Strip column names ---
    df.columns = df.columns.str.strip()
    
    # --- Pisahkan label ---
    label_col = [c for c in df.columns if 'label' in c.lower()]
    LABEL = label_col[0] if label_col else 'Label'
    y_raw = df[LABEL].astype(str).str.strip()
    
    # Remove rows where label is the header itself (embedded header rows)
    header_mask = y_raw.str.lower() == 'label'
    if header_mask.sum() > 0:
        print(f'  Removing {header_mask.sum()} embedded header rows')
        df = df[~header_mask].reset_index(drop=True)
        y_raw = y_raw[~header_mask].reset_index(drop=True)
    
    df = df.drop(columns=[LABEL])
    
    # --- Drop kolom Timestamp, Dst Port, Protocol + semua kolom object ---
    drop_lower = [x.lower() for x in DROP_COLS]
    cols_to_drop = [c for c in df.columns if c.lower() in drop_lower]
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = list(set(cols_to_drop + obj_cols))
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f'  Dropped columns ({len(cols_to_drop)}): {cols_to_drop[:10]}')
    stats['after_drop_cols'] = df.shape
    
    # --- Convert semua kolom ke numeric ---
    df = df.apply(pd.to_numeric, errors='coerce')
    
    # --- Handle Inf: replace dengan max finite value per kolom ---
    inf_count = np.isinf(df.values).sum()
    if inf_count > 0:
        for col in df.columns:
            mask_inf = np.isinf(df[col])
            if mask_inf.any():
                finite_vals = df[col][~mask_inf & df[col].notna()]
                if len(finite_vals) > 0:
                    max_val = finite_vals.max()
                    df[col] = df[col].replace([np.inf, -np.inf], max_val)
                else:
                    df[col] = df[col].replace([np.inf, -np.inf], 0)
        print(f'  Inf values imputed: {inf_count}')
    stats['inf_count'] = inf_count
    
    # --- Handle NaN: fill with median if too many, else drop ---
    nan_rows = df.isna().any(axis=1).sum()
    if nan_rows > 0:
        if nan_rows > len(df) * 0.5:
            print(f'  WARNING: {nan_rows} NaN rows ({nan_rows/len(df)*100:.1f}%) — filling with median')
            df = df.fillna(df.median())
        else:
            valid_idx = ~df.isna().any(axis=1)
            df = df[valid_idx].reset_index(drop=True)
            y_raw = y_raw[valid_idx].reset_index(drop=True)
            print(f'  NaN rows removed: {nan_rows}')
    stats['nan_rows_removed'] = nan_rows
    stats['after_nan_removal'] = df.shape
    print(f'  After NaN handling: {df.shape}')
    
    # --- Hapus kolom dengan varians nol ---
    if fit:
        variances = df.var()
        zero_var_cols = variances[variances == 0].index.tolist()
    if zero_var_cols:
        # Hanya drop kolom yang ada di df
        cols_exist = [c for c in zero_var_cols if c in df.columns]
        df = df.drop(columns=cols_exist)
        print(f'  Zero-variance columns removed ({len(cols_exist)}): {cols_exist[:10]}...' if len(cols_exist) > 10 else f'  Zero-variance columns removed ({len(cols_exist)}): {cols_exist}')
    stats['zero_var_cols'] = len(zero_var_cols) if zero_var_cols else 0
    stats['after_zero_var'] = df.shape
    
    # --- Simpan nama fitur ---
    feature_names = df.columns.tolist()
    
    # --- Scaling (StandardScaler) ---
    if fit:
        scaler = StandardScaler()
        X = scaler.fit_transform(df)
    else:
        X = scaler.transform(df)
    print(f'  Scaling applied (StandardScaler)')
    
    # --- Label Encoding ---
    if fit:
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y_raw)
    else:
        # Handle unseen labels gracefully
        known_classes = set(label_encoder.classes_)
        y_raw_clean = y_raw.apply(lambda x: x if x in known_classes else 'UNKNOWN')
        if 'UNKNOWN' not in known_classes:
            label_encoder.classes_ = np.append(label_encoder.classes_, 'UNKNOWN')
        y = label_encoder.transform(y_raw_clean)
    
    label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
    print(f'  Labels encoded: {len(label_mapping)} classes')
    stats['final_shape'] = X.shape
    stats['n_classes'] = len(label_mapping)
    
    return {
        'X': X,
        'y': y,
        'scaler': scaler,
        'label_encoder': label_encoder,
        'zero_var_cols': zero_var_cols,
        'label_mapping': label_mapping,
        'feature_names': feature_names,
        'stats': stats
    }

print('✓ Fungsi preprocess_dataset() berhasil didefinisikan.')

## 2. Proses file_100.csv (Referensi Utama)

File_100 diproses pertama kali sebagai referensi — scaler, label_encoder, dan zero_var_cols akan di-reuse untuk file lainnya agar konsisten.

In [ ]:
print('='*70)
print('Processing: file_100.csv (reference dataset)')
print('='*70)

df_100 = pd.read_csv(os.path.join(DATA_DIR, 'file_100.csv'), low_memory=False)
print(f'Loaded: {len(df_100):,} rows')

result_100 = preprocess_dataset(df_100, fit=True)

# Simpan referensi
ref_scaler = result_100['scaler']
ref_label_encoder = result_100['label_encoder']
ref_zero_var_cols = result_100['zero_var_cols']
ref_feature_names = result_100['feature_names']

# Save
output_data = {
    'X': result_100['X'],
    'y': result_100['y'],
    'feature_names': result_100['feature_names'],
    'label_mapping': result_100['label_mapping'],
    'scaler': ref_scaler,
    'label_encoder': ref_label_encoder,
    'stats': result_100['stats']
}
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'wb') as f:
    pickle.dump(output_data, f)

print(f'\nSaved: cleaned_100.pkl')
print(f'  X shape: {result_100["X"].shape}')
print(f'  y shape: {result_100["y"].shape}')
print(f'  Features: {len(ref_feature_names)}')
print(f'  Classes: {result_100["label_mapping"]}')

del df_100
gc.collect()

## 3. Proses file_75, file_50, file_25 (Menggunakan Referensi dari file_100)

In [ ]:
for input_file, output_file in zip(INPUT_FILES[1:], OUTPUT_FILES[1:]):
    print(f'\n{"="*70}')
    print(f'Processing: {input_file}')
    print(f'{"="*70}')
    
    filepath = os.path.join(DATA_DIR, input_file)
    if not os.path.exists(filepath):
        print(f'  FILE NOT FOUND — skipping')
        continue
    
    df = pd.read_csv(filepath, low_memory=False)
    print(f'  Loaded: {len(df):,} rows')
    
    result = preprocess_dataset(
        df, 
        scaler=ref_scaler, 
        label_encoder=ref_label_encoder,
        zero_var_cols=ref_zero_var_cols,
        fit=False
    )
    
    # Save
    output_data = {
        'X': result['X'],
        'y': result['y'],
        'feature_names': result['feature_names'],
        'label_mapping': result['label_mapping'],
        'scaler': ref_scaler,
        'label_encoder': ref_label_encoder,
        'stats': result['stats']
    }
    with open(os.path.join(DATA_DIR, output_file), 'wb') as f:
        pickle.dump(output_data, f)
    
    print(f'  Saved: {output_file}')
    print(f'    X shape: {result["X"].shape}')
    print(f'    y shape: {result["y"].shape}')
    
    del df, result
    gc.collect()

print(f'\n{"="*70}')
print('ALL FILES PROCESSED!')
print(f'{"="*70}')

## 4. Ringkasan & Verifikasi

In [ ]:
print(f'\n{"="*70}')
print(f'{"RINGKASAN PREPROCESSING":^70}')
print(f'{"="*70}')
print(f'\nReferensi (dari file_100):')
print(f'  Scaler: StandardScaler (mean/std fitted on file_100)')
print(f'  Label Encoder classes: {list(ref_label_encoder.classes_)}')
print(f'  Zero-variance cols removed: {len(ref_zero_var_cols)}')
print(f'  Final feature count: {len(ref_feature_names)}')
print(f'\nLabel Mapping:')
for label, idx in sorted(result_100['label_mapping'].items(), key=lambda x: x[1]):
    print(f'  {idx:>2d} → {label}')

print(f'\nOutput Files:')
for fname in OUTPUT_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'  {fname:20s} ({size_mb:.1f} MB)')
    else:
        print(f'  {fname:20s} (NOT FOUND)')

print(f'\nFeature Names ({len(ref_feature_names)}):')
for i, feat in enumerate(ref_feature_names):
    print(f'  [{i:>2d}] {feat}')

print(f'\n{"="*70}')
print('DONE! Cleaned data ready for model training.')
print(f'{"="*70}')

In [ ]:
# Quick sanity check — load cleaned_100 dan verifikasi
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    check = pickle.load(f)

print('Sanity check cleaned_100.pkl:')
print(f'  X type: {type(check["X"])} | shape: {check["X"].shape}')
print(f'  y type: {type(check["y"])} | shape: {check["y"].shape}')
print(f'  X mean ≈ 0? {check["X"].mean():.6f}')
print(f'  X std ≈ 1? {check["X"].std():.6f}')
print(f'  y unique values: {np.unique(check["y"])}')
print(f'  NaN in X: {np.isnan(check["X"]).sum()}')
print(f'  Inf in X: {np.isinf(check["X"]).sum()}')
print(f'  Label mapping: {check["label_mapping"]}')
print('\n✓ All checks passed!')